# M5 Monthly SKU Demand Forecasting — Debug Notebook

Replica o pipeline de `src/main.py` passo a passo para testar a execução e inspecionar resultados intermediários.

```bash
uv run jupyter notebook notebooks/main.ipynb
```

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from models.neural import NeuralModel
from train.loop import train_model
from data.loader import load_sales_data, validate_sales_schema
from evaluation.metrics import evaluate
from preprocess.scaler import scale_features, scale_target, unscale_target
from preprocess.transform import (
    MONTH_COLUMN,
    TARGET_AHEAD_COLUMN,
    build_dataset,
    preprocess_series,
)
from utils.pytorch import get_device, make_loader, predict
from preprocess.split import time_based_split
from utils.config import default_config

pd.set_option("display.max_columns", 20)
print(f"Project root: {PROJECT_ROOT}")
print(f"PyTorch: {torch.__version__}")

Project root: /Users/alissonrodrigues/personal/forecasting-project
PyTorch: 2.13.0


## Step 1 — Configuration

In [2]:
config = default_config()
config

Loading default project configuration...


ProjectConfig(lags=(1, 2, 3, 6, 12), forecast_horizon=1, train_end='2015-06', val_end='2015-12', target_column='sales', sku_column='item_id', date_column='date', data_path=PosixPath('/Users/alissonrodrigues/personal/forecasting-project/data/m5/processed/m5_daily_sales.parquet'), batch_size=32, epochs=10, learning_rate=0.001)

## Step 2 — Load and validate data

In [3]:
raw_data = load_sales_data(str(config.data_path))
data = validate_sales_schema(raw_data)

print(f"Rows: {len(data):,} | Columns: {list(data.columns)}")
data.head()

Loading sales data: /Users/alissonrodrigues/personal/forecasting-project/data/m5/processed/m5_daily_sales.parquet
  → Loaded 933648 rows, 4 columns
Validating sales data schema...
  → Schema validated successfully
Rows: 933,648 | Columns: ['date', 'item_id', 'store_id', 'sales']


,date,item_id,store_id,sales
0,2011-01-29,FOODS_1_218,CA_1,10.0
1,2011-01-30,FOODS_1_218,CA_1,3.0
2,2011-01-31,FOODS_1_218,CA_1,4.0
3,2011-02-01,FOODS_1_218,CA_1,0.0
4,2011-02-02,FOODS_1_218,CA_1,0.0


## Step 3 — Monthly time series per SKU

In [4]:
monthly = preprocess_series(data)

print(f"Monthly rows: {len(monthly):,} | SKUs: {monthly['item_id'].nunique()}")
monthly.head(10)

Preprocessing daily sales into monthly time series per SKU...
  → Monthly series: 3148 records (50 SKUs)
Monthly rows: 3,148 | SKUs: 50


,item_id,month,sales
0,FOODS_1_218,2011-01-01,149.0
1,FOODS_1_218,2011-02-01,663.0
2,FOODS_1_218,2011-03-01,1085.0
3,FOODS_1_218,2011-04-01,942.0
4,FOODS_1_218,2011-05-01,1050.0
5,FOODS_1_218,2011-06-01,1477.0
6,FOODS_1_218,2011-07-01,3689.0
7,FOODS_1_218,2011-08-01,2809.0
8,FOODS_1_218,2011-09-01,4459.0
9,FOODS_1_218,2011-10-01,4270.0


## Step 4 — Build dataset with lag features

In [5]:
supervised, lag_columns = build_dataset(
    monthly,
    lags=config.lags,
    horizon=config.forecast_horizon,
)

print(f"Dataset rows: {len(supervised):,}")
print(f"Lag columns: {lag_columns}")
supervised.head()

Building dataset (X, y) with horizon=1, lags=(1, 2, 3, 6, 12)...
  → Dataset: 2498 samples
Dataset rows: 2,498
Lag columns: ['lag_1', 'lag_2', 'lag_3', 'lag_6', 'lag_12']


,item_id,month,sales,lag_1,lag_2,lag_3,lag_6,lag_12,target
0,FOODS_1_218,2012-01-01,5312.0,3470.0,3351.0,4270.0,3689.0,149.0,7491.0
1,FOODS_1_218,2012-02-01,7491.0,5312.0,3470.0,3351.0,2809.0,663.0,6835.0
2,FOODS_1_218,2012-03-01,6835.0,7491.0,5312.0,3470.0,4459.0,1085.0,5167.0
3,FOODS_1_218,2012-04-01,5167.0,6835.0,7491.0,5312.0,4270.0,942.0,4832.0
4,FOODS_1_218,2012-05-01,4832.0,5167.0,6835.0,7491.0,3351.0,1050.0,6517.0


## Step 5 — Time-based split

In [6]:
(X_train, y_train), (X_val, y_val), (X_test, y_test) = time_based_split(
    supervised,
    date_column=MONTH_COLUMN,
    feature_columns=lag_columns,
    target_column=TARGET_AHEAD_COLUMN,
    train_end=config.train_end,
    val_end=config.val_end,
)

for name, X, y in [
    ("train", X_train, y_train),
    ("val", X_val, y_val),
    ("test", X_test, y_test),
]:
    print(f"{name:5} X={X.shape}  y={y.shape}  y mean={y.mean():.2f}  y std={y.std():.2f}")

Time-based split (train until 2015-06, validation until 2015-12)...
  → Train: X=(1948, 5), y=(1948,) | Val: X=(300, 5), y=(300,) | Test: X=(250, 5), y=(250,)
train X=(1948, 5)  y=(1948,)  y mean=4552.65  y std=3357.29
val   X=(300, 5)  y=(300,)  y mean=3919.31  y std=3060.59
test  X=(250, 5)  y=(250,)  y mean=3858.75  y std=2781.54


## Step 6 — Scale features and target

In [7]:
X_train_pp, X_val_pp, X_test_pp = scale_features(
    X_train, X_val, X_test, method="standardize"
)
y_train_s, y_val_s, y_test_s, y_loc, y_scale = scale_target(y_train, y_val, y_test)

print(f"X_train_pp mean={X_train_pp.mean():.4f} std={X_train_pp.std():.4f}")
X_train_pp[:3]

Scaling features with standardize (fit on train, apply to val/test)...
  → Train X=(1948, 5) mean=0.0000 std=1.0000
Scaling target with standardize (fit on train, apply to val/test)...
  → Train y mean=0.0000 std=1.0000
X_train_pp mean=0.0000 std=1.0000


array([[-0.3219794 , -0.36020467, -0.10671188, -0.28736273, -1.2493216 ],
       [ 0.21471715, -0.32599608, -0.36767482, -0.53472915, -1.10471727],
       [ 0.8496041 ,  0.20351828, -0.3338831 , -0.07091712, -0.98599543]])

## Step 7 — PyTorch data loaders

In [8]:
device = get_device()
train_loader = make_loader(
    X_train_pp, y_train_s, batch_size=config.batch_size, shuffle=True
)
val_loader = make_loader(
    X_val_pp, y_val_s, batch_size=config.batch_size, shuffle=False
)

print(f"Device: {device}")
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

Device: mps
Train batches: 61 | Val batches: 10


## Step 8 — Train, save, and evaluate

In [9]:
model = NeuralModel(input_dim=X_train_pp.shape[1], hidden_dim=64, output_dim=1)

print("─── Starting Neural Network Training ───")
model = train_model(
    model,
    train_loader,
    val_loader,
    device=device,
    epochs=config.epochs,
    learning_rate=config.learning_rate,
)

─── Starting Neural Network Training ───
  Training on device: mps
Epoch 1/10 - Train MSE: 0.6112 | Val MSE: 0.2379
Epoch 2/10 - Train MSE: 0.3098 | Val MSE: 0.2269
Epoch 3/10 - Train MSE: 0.2961 | Val MSE: 0.2343
Epoch 4/10 - Train MSE: 0.2886 | Val MSE: 0.2430
Epoch 5/10 - Train MSE: 0.2802 | Val MSE: 0.2436
Epoch 6/10 - Train MSE: 0.2777 | Val MSE: 0.2524
Epoch 7/10 - Train MSE: 0.2726 | Val MSE: 0.2571
Epoch 8/10 - Train MSE: 0.2707 | Val MSE: 0.2479
Epoch 9/10 - Train MSE: 0.2692 | Val MSE: 0.2519
Epoch 10/10 - Train MSE: 0.2679 | Val MSE: 0.2484


In [10]:
model_path = PROJECT_ROOT / "demand_model.pth"
torch.save(model.state_dict(), model_path)
print(f"Model saved to {model_path}")

Model saved to /Users/alissonrodrigues/personal/forecasting-project/demand_model.pth


In [11]:
val_predictions = unscale_target(predict(model, X_val_pp, device), y_loc, y_scale)
test_predictions = unscale_target(predict(model, X_test_pp, device), y_loc, y_scale)

val_metrics = evaluate(y_val, val_predictions, split="validation")
test_metrics = evaluate(y_test, test_predictions, split="test")

Evaluating validation performance...
  → MAE:  1079.5066  (mean y=3919.3133, MAE/mean=0.2754)
  → RMSE: 1673.1601
Evaluating test performance...
  → MAE:  981.9185  (mean y=3858.7480, MAE/mean=0.2545)
  → RMSE: 1429.9419


In [12]:
print("=" * 60)
print(" Final Summary")
print("=" * 60)
print(f"  Training samples:    {X_train_pp.shape[0]}")
print(f"  Validation samples:  {X_val_pp.shape[0]}")
print(f"  Test samples:        {X_test_pp.shape[0]}")
print(f"  Features (lags):     {X_train_pp.shape[1]}")
print(f"  Validation MAE:      {val_metrics['mae']:.4f}  (MAE/mean={val_metrics['mae_over_mean']:.4f})")
print(f"  Validation RMSE:     {val_metrics['rmse']:.4f}")
print(f"  Test MAE:            {test_metrics['mae']:.4f}  (MAE/mean={test_metrics['mae_over_mean']:.4f})")
print(f"  Test RMSE:           {test_metrics['rmse']:.4f}")
print("=" * 60)
print("\nPipeline finished successfully.")

 Final Summary
  Training samples:    1948
  Validation samples:  300
  Test samples:        250
  Features (lags):     5
  Validation MAE:      1079.5066  (MAE/mean=0.2754)
  Validation RMSE:     1673.1601
  Test MAE:            981.9185  (MAE/mean=0.2545)
  Test RMSE:           1429.9419

Pipeline finished successfully.


In [13]:
# Summary printed in the cell above.